<a href="https://colab.research.google.com/github/Stdcoders/Graph-RAG/blob/main/GraphRAG_L8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 403 (delta 125), reused 57 (delta 55), pack-reused 224 (from 2)
Receiving objects: 100% (403/403), 13.34 MiB | 21.44 MiB/s, done.
Resolving deltas: 100% (217/217), done.


In [ ]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.0/358.0 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━

In [ ]:
import os

os.chdir("/content/Agentic_KAG_Workshop_DHS_2026")

from dotenv import load_dotenv

load_dotenv()  # This loads .env at project root

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NVIDIA_API_KEY = os.getenv('NVIDIA_API_KEY')

# Set OPENAI_API_KEY as env variable for openai/neo4j-graphrag compatibility
os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# (Optional) Test the connection
driver.verify_connectivity()

In [ ]:
import os
from typing import List
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings.openai import BaseOpenAIEmbeddings
from langchain_core.embeddings import Embeddings  # <-- for LangChain compatibility

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

class NemotronEmbeddings(BaseOpenAIEmbeddings, Embeddings):
    """
    NVIDIA Nemotron embeddings via NIM's OpenAI-compatible /v1/embeddings endpoint.
    Implements BOTH the neo4j_graphrag Embedder interface (embed_query) AND
    the LangChain Embeddings interface (embed_documents + embed_query),
    so the same object works in the KG pipeline AND in Chroma.from_documents.
    """
    def __init__(self, model="nvidia/nemotron-3-embed-1b", input_type="passage", **kwargs):
        self.input_type = input_type
        super().__init__(model=model, **kwargs)

    def _initialize_client(self, **kwargs):
        return self.openai.OpenAI(**kwargs)

    def embed_query(self, text, **kwargs):
        kwargs.setdefault("extra_body", {"input_type": self.input_type})
        return super().embed_query(text, **kwargs)

    def embed_documents(self, texts: List[str], batch_size: int = 32, **kwargs) -> List[List[float]]:
      kwargs.setdefault("extra_body", {"input_type": "passage"})
      all_embeddings = []
      for i in range(0, len(texts), batch_size):
          batch = texts[i:i + batch_size]
          response = self.client.embeddings.create(input=batch, model=self.model, **kwargs)
          all_embeddings.extend([d.embedding for d in response.data])
          print(f"Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks")
      return all_embeddings

neo4j_llm = OpenAILLM(
    model_name="nvidia/nemotron-3-super-120b-a12b",
    model_params={"response_format": {"type": "json_object"}},
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY,
)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=NVIDIA_API_KEY,
    openai_api_base=NVIDIA_BASE_URL,
    model_name="nvidia/nemotron-3-super-120b-a12b",
    temperature=0,
)

embedder = NemotronEmbeddings(
    model="nvidia/nemotron-3-embed-1b",
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY,
)

query_embedder = NemotronEmbeddings(
    model="nvidia/nemotron-3-embed-1b",
    input_type="query",
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY,
)
embeddings = NemotronEmbeddings(
    model="nvidia/nemotron-3-embed-1b",
    input_type="query",  # Chroma's embed_documents() ignores this and always sends "passage" internally,
                          # so "query" here just governs what embed_query() uses at retrieval time
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY,
)
print('LLM + embedders ready ✔')

LLM + embedders ready ✔


In [ ]:
from langfuse.langchain import CallbackHandler
from langfuse import get_client

os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

langfuse_handler = CallbackHandler()

Langfuse client is authenticated and ready!


In [ ]:
# Rebuild text_chunks in-place (no dependency on a pre-saved .pkl file)
import ast
from pathlib import Path
import pandas as pd
from neo4j_graphrag.experimental.components.data_loader import DataLoader
from neo4j_graphrag.experimental.components.text_splitters.base import TextSplitter
from neo4j_graphrag.experimental.components.types import (
    TextChunks, TextChunk, LoadedDocument
)

excel_path = "/content/Agentic_KAG_Workshop_DHS_2026/data/TMDB_IMDB_Movies_Dataset_filtered.xlsx"
DATA_PATH = Path(excel_path)

class MoviesDataLoader(DataLoader):
    def __init__(self, **kwargs):
        pass

    async def run(self, path: Path) -> LoadedDocument:
        df = pd.read_excel(path).reset_index(drop=True)
        if 'release_date' in df.columns:
            df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
            df['release_date'] = df['release_date'].dt.strftime('%Y-%m-%d').replace({pd.NA: None})
        df = df.where(pd.notna(df), None)
        records = df.to_dict(orient='records')
        return LoadedDocument(text=repr(records), document_info={'path': str(path)})

def _row_to_movie_doc(rec: dict) -> str:
    display_fields = {
        'title': {'label': 'Title'},
        'release_date': {'label': 'Release date'},
        'runtime': {'label': 'Runtime (minutes)', 'formatter': lambda x: f"{x:.0f}"},
        'budget': {'label': 'Budget (USD)', 'formatter': lambda x: f"{x:,.0f}"},
        'revenue': {'label': 'Revenue (USD)', 'formatter': lambda x: f"{x:,.0f}"},
        'vote_average': {'label': 'Vote average', 'formatter': lambda x: f"{x:.1f}"},
        'genres': {'label': 'Genres'},
        'production_companies': {'label': 'Production companies'},
        'production_countries': {'label': 'Production countries'},
        'spoken_languages': {'label': 'Spoken languages'},
        'keywords': {'label': 'Keywords'},
        'directors': {'label': 'Director(s)'},
        'cast': {'label': 'Cast'},
        'tagline': {'label': 'Tagline'},
        'overview': {'label': 'Overview'},
    }
    lines = []
    for key, config in display_fields.items():
        value = rec.get(key)
        if value is not None and value != '' and not (isinstance(value, (list, tuple, dict)) and not value):
            formatter = config.get('formatter')
            try:
                formatted_value = formatter(value) if formatter else str(value)
            except (TypeError, ValueError):
                formatted_value = str(value)
            lines.append(f"{config['label']}: {formatted_value}")
    return '\n'.join(lines)

class MoviesRowTextSplitter(TextSplitter):
    def __init__(self, dataset_name: str = 'TMDB+IMDb Movies'):
        self.dataset_name = dataset_name

    async def run(self, page_data: LoadedDocument) -> TextChunks:
        raw = page_data['text'] if isinstance(page_data, dict) else page_data.text
        records = ast.literal_eval(raw)
        chunks = [
            TextChunk(
                index=i,
                text=_row_to_movie_doc(rec),
                metadata={
                    'title': rec.get('title', ''),
                    'release_date': rec.get('release_date', ''),
                    'dataset': self.dataset_name,
                },
            )
            for i, rec in enumerate(records)
        ]
        return TextChunks(chunks=chunks)

# Execute
loader = MoviesDataLoader()
page_data = await loader.run(DATA_PATH)
splitter = MoviesRowTextSplitter()
text_chunks = await splitter.run(page_data)
print(f"Rebuilt {len(text_chunks.chunks)} text chunks.")

Rebuilt 4869 text chunks.


In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=chunk.text,
        metadata={
            **chunk.metadata,
            "uid": chunk.uid,
            "chunk_index": chunk.index,
        },
    )
    for chunk in text_chunks.chunks
]
print(f"Created {len(documents)} LangChain documents.")

Created 4869 LangChain documents.


In [ ]:
print(type(documents[0]))
print(documents[0].page_content[:200])
print(documents[0].metadata)


<class 'langchain_core.documents.base.Document'>
Title: Eternals
Release date: 2021-11-03
Runtime (minutes): 156
Budget (USD): 200,000,000
Revenue (USD): 402,064,899
Vote average: 6.9
Genres: Science Fiction, Action, Adventure
Production companies: 
{'title': 'Eternals', 'release_date': '2021-11-03', 'dataset': 'TMDB+IMDb Movies', 'uid': 'c349aa70-87fc-4fa7-90db-b81e337e5f25', 'chunk_index': 0}


In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.documents import Document
from urllib.parse import urlparse
from langchain_community.vectorstores import Chroma
from langchain_classic.prompts import ChatPromptTemplate
from IPython.display import display, Markdown

persist_directory = os.getcwd() +'/vectorstore/chroma/'

# Create the vector store
vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(vectordb._collection.count())

# Setup
similarity_search_retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": 5})
bm25_retriever = BM25Retriever.from_documents(documents=documents, k=5)
ensemble_retriever = EnsembleRetriever(retrievers=[similarity_search_retriever, bm25_retriever], weights=[0.5, 0.5])


template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Avoid using generic phrases like "Provide context" or "as per context.

Question: {input}

Context: {context}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

# Combine docs and chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(ensemble_retriever, combine_docs_chain)

def run_traditional_rag(question):
    response = rag_chain.invoke({"input": question}, config={"callbacks": [langfuse_handler]})
    return response["answer"]

/tmp/ipykernel_1432/4257186350.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Embedded 32/4869 chunks
Embedded 64/4869 chunks
Embedded 96/4869 chunks
Embedded 128/4869 chunks
Embedded 160/4869 chunks
Embedded 192/4869 chunks
Embedded 224/4869 chunks
Embedded 256/4869 chunks
Embedded 288/4869 chunks
Embedded 320/4869 chunks
Embedded 352/4869 chunks
Embedded 384/4869 chunks
Embedded 416/4869 chunks
Embedded 448/4869 chunks
Embedded 480/4869 chunks
Embedded 512/4869 chunks
Embedded 544/4869 chunks
Embedded 576/4869 chunks
Embedded 608/4869 chunks
Embedded 640/4869 chunks
Embedded 672/4869 chunks
Embedded 704/4869 chunks
Embedded 736/4869 chunks
Embedded 768/4869 chunks
Embedded 800/4869 chunks
Embedded 832/4869 chunks
Embedded 864/4869 chunks
Embedded 896/4869 chunks
Embedded 928/4869 chunks
Embedded 960/4869 chunks
Embedded 992/4869 chunks
Embedded 1024/4869 chunks
Embedded 1056/4869 chunks
Embedded 1088/4869 chunks
Embedded 1120/4869 chunks
Embedded 1152/4869 chunks
Embedded 1184/4869 chunks
Embedded 1216/4869 chunks
Embedded 1248/4869 chunks
Embedded 1280/4869 c

In [ ]:
from langchain_classic.agents import create_react_agent, AgentExecutor
import prompts
from langchain_classic.tools import Tool
from langchain_classic.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    PromptTemplate,
)
retriever_tool = Tool(
    name="AV_Agentic_RAG_tool",
    description="Useful for retrieving relevant documents based on input questions.",
    func=lambda query: ensemble_retriever.invoke(query),
)
tools = [retriever_tool]

# Get the ReAct prompt
prompt = prompts.REACT_PROMPT


# Create the ReAct agent
def get_react_agent(llm, tools, system_prompt, verbose=False):
    """Helper function for creating agent executor"""
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="conversation_history", optional=True),
            HumanMessagePromptTemplate(
                prompt=PromptTemplate(input_variables=["input"], template="{input}")
            ),
            AIMessagePromptTemplate(
                prompt=PromptTemplate(
                    input_variables=["agent_scratchpad"], template="{agent_scratchpad}"
                )
            ),
        ]
    )
    agent = create_react_agent(llm, tools, prompt)
    return AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=verbose,
        stream_runnable=True,
        handle_parsing_errors=True,
        max_iterations=5,
        return_intermediate_steps=True,
    )

generate_agent = get_react_agent(
        llm,
        tools,
        prompt,
        verbose=False,
    )


def run_agentic_rag(question):
    answer = generate_agent.invoke({"input": question}, config={"callbacks": [langfuse_handler]})
    return answer["output"]
